# 05 - Invoice Extraction and Evaluation

This notebook runs the rule-based invoice information extraction pipeline on documents classified as invoices.

It covers:
- Why extraction needs to be layout-aware, not just regex-based
- How spatial zone detection improves field selection
- Running extraction on ground-truth invoice documents from the test split
- Qualitative analysis of results and failure cases
- Creating a manual annotation template for evaluation
- Per-field success rate reporting

## Why layout awareness matters

A naive regex approach would scan the full OCR text of an invoice and return the first match for each field pattern. This fails in practice for several reasons:

- Invoices often contain multiple dates: the invoice date, due date, and sometimes order or shipping dates. A regex with no spatial context will pick whichever comes first in the text, which may be wrong.
- The word 'total' appears multiple times on most invoices: subtotal, tax total, shipping total, and grand total. Without knowing which amount sits in the totals zone at the bottom right of the page, a regex will often return the subtotal instead of the final amount.
- Company names are not reliably labeled. The issuer name is usually the first prominent line on the page, not preceded by any keyword. A regex cannot find it without spatial context.
- OCR text is read in document order, which means lines from the left column and right column are interleaved. A pure text scan loses the spatial relationships between labels and values.

Our approach divides the page into four zones based on OCR bounding box coordinates: seller (top-left), metadata (top-right), buyer (mid-left introduced by bill-to anchors), and totals (bottom-right). Each field extractor is directed to search the most relevant zone first before falling back to the full text.

## Setup

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')

# Make sure the project root is on the path so src imports work.
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('Project root:', PROJECT_ROOT)

In [ ]:
from src.ocr_engine import ocr_document, load_ocr_result, check_tesseract_installation
from src.invoice_extraction import extract_invoice_fields, extract_batch
from src.zones import detect_zones, zone_summary, get_lines_in_zone, lines_to_text
from src.validators import validate_date, validate_amount, compute_field_confidence
from src.config import OCRConfig, load_ocr_config

# Verify Tesseract is available before trying to run OCR.
tesseract_ok = check_tesseract_installation(verbose=True)
if not tesseract_ok:
    print('Tesseract not found. Install it with: brew install tesseract')
    print('OCR will load from cache if available, but cannot process new documents.')

In [ ]:
# Paths
DATA_DIR       = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR     = PROJECT_ROOT / 'outputs' / 'extraction'
ANNOTATION_DIR = PROJECT_ROOT / 'outputs' / 'extraction' / 'annotation'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_ocr_config(PROJECT_ROOT / 'configs' / 'config.yaml')

print('Output directory:', OUTPUT_DIR)

## Load invoice documents from the test split

We use ground-truth invoice labels from the test split for development and evaluation.
In the final pipeline, this notebook would receive predicted invoice doc_ids from the
best classifier (notebook 06). Both modes are supported below.

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print('Test split size:', len(test_df))
print('Classes in test split:')
print(test_df['class_name'].value_counts())

In [ ]:
# Filter to ground-truth invoices for development.
invoice_df = test_df[test_df['class_name'] == 'invoice'].copy()
invoice_df = invoice_df.reset_index(drop=True)

print(f'Invoice documents in test split: {len(invoice_df)}')
invoice_df.head()

## Run OCR on invoice documents

The OCR engine caches results to `data/interim/ocr/parsed/`. If a cache already
exists for a document it will not be reprocessed. We run on a sample first to
keep runtime manageable during development.

In [ ]:
# Use a sample for development. Set SAMPLE_SIZE to None to run on all invoices.
SAMPLE_SIZE = 50

if SAMPLE_SIZE is not None:
    sample_df = invoice_df.sample(n=min(SAMPLE_SIZE, len(invoice_df)), random_state=42)
else:
    sample_df = invoice_df

print(f'Running extraction on {len(sample_df)} invoice documents')

In [ ]:
from tqdm.notebook import tqdm

ocr_results = {}
ocr_errors  = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='OCR'):
    doc_id    = str(row['doc_id'])
    file_path = str(row['file_path'])

    try:
        result = ocr_document(doc_id, file_path, cfg=cfg, force=False)
        ocr_results[doc_id] = result
    except Exception as e:
        ocr_errors.append({'doc_id': doc_id, 'error': str(e)})

print(f'OCR succeeded: {len(ocr_results)}')
print(f'OCR errors:    {len(ocr_errors)}')

## Run field extraction

In [ ]:
extraction_results = []

for doc_id, ocr_result in tqdm(ocr_results.items(), desc='Extracting fields'):
    result = extract_invoice_fields(
        ocr_result,
        doc_id=doc_id,
        predicted_class='invoice',
        debug=False,
    )
    extraction_results.append(result)

extraction_df = pd.DataFrame(extraction_results)
print(f'Extraction complete: {len(extraction_df)} documents')
extraction_df.head()

## Save extraction results

In [ ]:
output_path = OUTPUT_DIR / 'invoice_extraction_results.csv'
extraction_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

## Per-field success rates

Since we do not have labeled ground truth for field values, we measure success
as whether a non-null value was extracted for each field. This is a lower bound
on true accuracy but gives a useful signal about which fields the extractor
struggles with most.

In [ ]:
fields = ['invoice_number', 'invoice_date', 'due_date', 'issuer_name', 'recipient_name', 'total_amount']

success_rates = {}
for field in fields:
    if field in extraction_df.columns:
        non_null = extraction_df[field].notna() & (extraction_df[field].astype(str).str.strip() != '') & (extraction_df[field].astype(str) != 'None')
        success_rates[field] = round(non_null.mean() * 100, 1)

success_df = pd.DataFrame([
    {'field': k, 'extraction_rate_pct': v} for k, v in success_rates.items()
])

print('Per-field extraction success rates:')
print(success_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(success_df['field'], success_df['extraction_rate_pct'], color='steelblue')
ax.set_xlabel('Extraction rate (%)')
ax.set_title('Per-field extraction success rate')
ax.set_xlim(0, 100)
for i, val in enumerate(success_df['extraction_rate_pct']):
    ax.text(val + 1, i, f'{val}%', va='center')
plt.tight_layout()

fig_path = PROJECT_ROOT / 'reports' / 'figures' / 'invoice_field_extraction_rates.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved figure to {fig_path}')

In [ ]:
# Confidence score distribution across all extracted documents.
fig, ax = plt.subplots(figsize=(7, 3))
extraction_df['extraction_confidence'].hist(bins=20, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Extraction confidence')
ax.set_ylabel('Number of documents')
ax.set_title('Distribution of extraction confidence scores')
plt.tight_layout()

conf_fig_path = PROJECT_ROOT / 'reports' / 'figures' / 'invoice_extraction_confidence_dist.png'
fig.savefig(conf_fig_path, dpi=140, bbox_inches='tight')
plt.show()

## Qualitative analysis: sample extractions

We show a few examples with the raw OCR text, detected zones, and extracted fields
side by side. This helps identify where the extractor succeeds and where it struggles.

In [ ]:
def show_extraction_example(doc_id: str, ocr_result: dict, extraction: dict):
    lines = ocr_result.get('lines', [])
    image_w = int(ocr_result.get('image_width', 1))
    image_h = int(ocr_result.get('image_height', 1))

    zones = detect_zones(lines, image_w, image_h)

    print('=' * 60)
    print(f'Document: {doc_id}')
    print(f'Zone summary: {zone_summary(zones)}')
    print()
    print('OCR text (first 20 lines):')
    for line in lines[:20]:
        print(f'  [{line.get("top", 0):4d}px] {line.get("text", "")}')
    print()
    print('Extracted fields:')
    fields_to_show = [
        'invoice_number', 'invoice_date', 'due_date',
        'issuer_name', 'recipient_name', 'total_amount',
        'extraction_confidence'
    ]
    for field in fields_to_show:
        print(f'  {field:30s}: {extraction.get(field)}')
    print()

In [ ]:
# Show up to 5 examples.
shown = 0
for row in extraction_results:
    doc_id = row['doc_id']
    if doc_id in ocr_results:
        show_extraction_example(doc_id, ocr_results[doc_id], row)
        shown += 1
    if shown >= 5:
        break

## Failure case analysis

Documents with a low confidence score are likely to have missing fields.
We inspect the lowest-confidence extractions to understand common failure patterns.

In [ ]:
low_confidence = extraction_df[extraction_df['extraction_confidence'] < 0.5].sort_values('extraction_confidence')
print(f'Documents with confidence below 0.5: {len(low_confidence)}')

# Show worst cases.
shown = 0
for _, row in low_confidence.iterrows():
    doc_id = row['doc_id']
    if doc_id in ocr_results:
        show_extraction_example(doc_id, ocr_results[doc_id], row.to_dict())
        shown += 1
    if shown >= 3:
        break

## Manual annotation template

Since the dataset has no field-level ground truth labels, we create an annotation
template CSV. A human annotator can open this file, fill in the correct values
for each document, and save it. The evaluation cell below then computes exact
match and normalized match metrics against those annotations.

In [ ]:
# Sample documents for annotation. We take a mix of high and low confidence.
ANNOTATION_SAMPLE_SIZE = 20

high_conf = extraction_df[extraction_df['extraction_confidence'] >= 0.6].sample(
    n=min(10, len(extraction_df[extraction_df['extraction_confidence'] >= 0.6])),
    random_state=42
)
low_conf = extraction_df[extraction_df['extraction_confidence'] < 0.6].sample(
    n=min(10, len(extraction_df[extraction_df['extraction_confidence'] < 0.6])),
    random_state=42
)

annotation_sample = pd.concat([high_conf, low_conf]).reset_index(drop=True)

# Build the annotation template with predicted values pre-filled and empty ground truth columns.
annotation_template = annotation_sample[['doc_id'] + fields].copy()
annotation_template.columns = ['doc_id'] + [f'predicted_{f}' for f in fields]

for f in fields:
    annotation_template[f'true_{f}'] = ''  # annotator fills these in

annotation_path = ANNOTATION_DIR / 'annotation_template.csv'
annotation_template.to_csv(annotation_path, index=False)

print(f'Annotation template saved to: {annotation_path}')
print(f'Documents to annotate: {len(annotation_template)}')
print()
print('Instructions for annotators:')
print('  1. Open annotation_template.csv in a spreadsheet application.')
print('  2. For each row, look up the source document using the doc_id.')
print('  3. Fill in the true_* columns with the correct field values.')
print('  4. Leave a cell empty if the field does not exist in that document.')
print('  5. Save as annotation_results.csv in the same folder.')

## Evaluation against annotations

Run the cell below after completing the annotation template.
If no annotation file exists yet, it will print a message and skip.

In [ ]:
import re

def normalize_for_comparison(value) -> str:
    """Normalize a field value for comparison by lowercasing and removing punctuation."""
    if value is None or str(value).strip() == '' or str(value) == 'nan':
        return ''
    return re.sub(r'[^a-z0-9]', '', str(value).lower().strip())


annotation_results_path = ANNOTATION_DIR / 'annotation_results.csv'

if not annotation_results_path.exists():
    print('No annotation results file found yet.')
    print(f'Complete the template at {annotation_path} and save it as annotation_results.csv')
else:
    ann_df = pd.read_csv(annotation_results_path)

    eval_rows = []
    for _, row in ann_df.iterrows():
        doc_id = row['doc_id']
        for f in fields:
            predicted = normalize_for_comparison(row.get(f'predicted_{f}', ''))
            true_val  = normalize_for_comparison(row.get(f'true_{f}', ''))

            if true_val == '':
                # Skip fields the annotator left blank.
                continue

            exact_match = int(predicted == true_val)
            # Partial match: predicted is a substring of true or vice versa.
            partial_match = int(
                predicted != '' and
                (predicted in true_val or true_val in predicted)
            )

            eval_rows.append({
                'doc_id': doc_id,
                'field': f,
                'exact_match': exact_match,
                'partial_match': partial_match,
            })

    eval_df = pd.DataFrame(eval_rows)

    summary = eval_df.groupby('field').agg(
        exact_match_rate=('exact_match', 'mean'),
        partial_match_rate=('partial_match', 'mean'),
        n_annotated=('exact_match', 'count'),
    ).reset_index()

    summary['exact_match_rate']   = (summary['exact_match_rate'] * 100).round(1)
    summary['partial_match_rate'] = (summary['partial_match_rate'] * 100).round(1)

    print('Evaluation results against manual annotations:')
    print(summary.to_string(index=False))

    eval_path = OUTPUT_DIR / 'evaluation_summary.csv'
    summary.to_csv(eval_path, index=False)
    print(f'\nEvaluation summary saved to {eval_path}')

## Integration with classifier predictions

In the final pipeline (notebook 06), the classifier produces a predictions CSV.
The cell below shows how to run extraction on those predicted invoices instead
of ground-truth labels.

In [ ]:
predictions_path = PROJECT_ROOT / 'outputs' / 'predictions' / 'final_predictions.csv'

if predictions_path.exists():
    pred_df = pd.read_csv(predictions_path)
    predicted_invoices = pred_df[pred_df['pred_label'] == 'invoice'].copy()
    print(f'Predicted invoices from classifier: {len(predicted_invoices)}')

    # Merge with metadata to get file paths.
    meta_df = pd.read_csv(DATA_DIR / 'test.csv')
    predicted_invoices = predicted_invoices.merge(meta_df[['doc_id', 'file_path']], on='doc_id', how='left')

    print('Running OCR and extraction on classifier-predicted invoices...')
    classifier_extraction_results = []

    for _, row in tqdm(predicted_invoices.iterrows(), total=len(predicted_invoices), desc='Classifier invoices'):
        doc_id    = str(row['doc_id'])
        file_path = str(row['file_path'])
        confidence = row.get('confidence_invoice', None)

        try:
            ocr_result = ocr_document(doc_id, file_path, cfg=cfg, force=False)
            result = extract_invoice_fields(
                ocr_result,
                doc_id=doc_id,
                predicted_class='invoice',
                predicted_confidence=confidence,
            )
        except Exception as e:
            result = {'doc_id': doc_id, 'error': str(e), 'extraction_confidence': 0.0}

        classifier_extraction_results.append(result)

    classifier_extraction_df = pd.DataFrame(classifier_extraction_results)
    classifier_output_path = OUTPUT_DIR / 'classifier_predicted_invoice_extractions.csv'
    classifier_extraction_df.to_csv(classifier_output_path, index=False)
    print(f'Saved to {classifier_output_path}')

else:
    print('No classifier predictions file found yet.')
    print(f'Expected at: {predictions_path}')
    print('Run this cell after notebook 06 has been completed.')

## Summary

This notebook demonstrated the full invoice extraction pipeline:

- Zone detection divides the page into seller, buyer, metadata, and totals regions using OCR bounding box coordinates.
- Each field extractor searches the most relevant zone first, using anchor keywords and regex patterns, before falling back to the full document text.
- Validators reject implausible candidates such as phone numbers masquerading as invoice numbers or subtotals being mistaken for grand totals.
- An extraction confidence score summarizes how many of the six required fields were successfully found.
- A manual annotation template was exported so field-level accuracy can be measured once annotations are available.

The main limitation of this approach is that it relies on OCR quality. Documents with poor scan quality, unusual layouts, or handwritten content will produce lower extraction confidence and more missing fields.